# GOTOken oracle

Reference values from the HuggingFace `SmolLM2-135M` model for each step's checkpoint,
and comparisons against the BASIC engine. All logic lives in `oracle.py`; this notebook
is the interactive front-end. Run `./build.sh` in the repo root first if you want the
BASIC comparisons.

In [1]:
from oracle import *
tok = load_tokenizer()
model = load_model()
model.config

/Users/bmuskalla/git/GOTOken/export/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 7036.82it/s]

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "float32",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "use_cache": true,
  "vocab_size": 49152
}

## Step 2: embedding row -> tied output head

No norm, no attention, no FFN. With tied weights every logit is the dot product of the
input token's embedding with one vocab row, so the "prediction" is simply the nearest
embeddings. Note what the top-5 for ` cat` looks like.

In [2]:
for i in tok.encode("The cat sat", add_special_tokens=False):
    print(i, repr(tok.convert_ids_to_tokens(i)))

504 'The'
2644 'Ġcat'
2643 'Ġsat'


In [3]:
_ = print_step2(model, tok, 2644)

token 2644 = 'Ġcat'
logit 0 0.133899
logit 1 1.771975
logit 2 1.769763
logit 3 1.899459
logit 4 1.86695
top 1 id 2644 logit 3.985926   'Ġcat'
top 2 id 9786 logit 3.531259   'cat'
top 3 id 40578 logit 3.432986   'cats'
top 4 id 27772 logit 3.333802   'Cat'
top 5 id 6 logit 3.297556   '<filename>'


In [4]:
compare_step2(model, tok, 2644)


compare BASIC vs oracle
  logit 0: basic 0.133899  oracle 0.133899  diff 6.12e-09  ok
  logit 1: basic 1.771975  oracle 1.771975  diff 4.50e-07  ok
  logit 2: basic 1.769764  oracle 1.769763  diff 7.33e-07  ok
  logit 3: basic 1.899459  oracle 1.899459  diff 3.95e-07  ok
  logit 4: basic 1.86695  oracle 1.86695  diff 1.35e-07  ok
  top 1: basic id 2644 3.985922  oracle id 2644 3.985926  diff 3.90e-06  ok
  top 2: basic id 9786 3.531259  oracle id 9786 3.531259  diff 1.81e-07  ok
  top 3: basic id 40578 3.432987  oracle id 40578 3.432986  diff 8.10e-07  ok
  top 4: basic id 27772 3.333801  oracle id 27772 3.333802  diff 8.67e-07  ok
  top 5: basic id 6 3.297555  oracle id 6 3.297556  diff 7.97e-07  ok
PASS


True